In [ ]:
# Install dependencies
!pip install -q torch torchvision scikit-learn scikit-image opencv-python tqdm pandas pillow

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

print('\n✓ Setup complete!')

In [ ]:
# Step 1: Locate repo/script in Kaggle/Colab inputs
import sys
from pathlib import Path

def find_train_script():
    search_roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path("/content")]
    hits = []
    for root in search_roots:
        if root.exists():
            hits.extend(root.rglob("train_hybrid.py"))

    if not hits:
        raise FileNotFoundError(
            "train_hybrid.py not found in /kaggle/input, /kaggle/working, or /content."
        )

    # Prefer the script that has feature_extractor.py beside it
    for p in hits:
        if (p.parent / "feature_extractor.py").exists():
            return p

    return hits[0]

script_path = find_train_script()
REPO_DIR = script_path.parent

for p in (REPO_DIR, REPO_DIR.parent):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f"✓ Script found at: {script_path}")
print(f"✓ Repo/code dir: {REPO_DIR}")
print("✓ Setup complete!")

In [ ]:
# Step 2: Verify Data (Kaggle/Colab-safe)
import os
import shutil
from pathlib import Path

print("📋 Checking for data...\n")

local_data_path = Path("data")

def find_dataset_root():
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path("/content")]
    for root in roots:
        if not root.exists():
            continue
        for train_dir in root.rglob("train"):
            if train_dir.is_dir() and (train_dir.parent / "val").is_dir():
                return train_dir.parent
    return None

dataset_path = find_dataset_root()

if dataset_path:
    print(f"✅ Found dataset at: {dataset_path}\n")

    if not local_data_path.exists():
        print("📦 Copying data to working directory...")
        shutil.copytree(dataset_path, local_data_path)
        print("✅ Data copied!")

    print("\n📊 Data structure:")
    for root, dirs, files in os.walk(local_data_path):
        level = root.replace(str(local_data_path), "").count(os.sep)
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = " " * 2 * (level + 1)
        file_count = len([f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))])
        if file_count > 0:
            print(f"{subindent}{file_count} images")

    print("\n✓ Data ready for training!")
else:
    raise FileNotFoundError("❌ Could not find a dataset with train/ and val/ folders.")

In [ ]:
# Verify data counts
import os

def count_images(directory):
    classes = {}
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            count = len([f for f in os.listdir(class_path) if f.endswith(('.png', '.jpg'))])
            classes[class_name] = count
    return classes

print('Training Data:')
train_counts = count_images('data/train')
for cls, count in sorted(train_counts.items()):
    print(f'  {cls}: {count}')

print('\nValidation Data:')
val_counts = count_images('data/val')
for cls, count in sorted(val_counts.items()):
    print(f'  {cls}: {count}')

print(f'\nTotal Training: {sum(train_counts.values())}')
print(f'Total Validation: {sum(val_counts.values())}')
print('\n✓ Data verified!')

In [ ]:
import sys, os, subprocess, shutil
from pathlib import Path

def find_train_script_in_roots(roots):
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob("backend/scripts/train_hybrid.py"):
            return p
        for p in root.rglob("train_hybrid.py"):
            return p
    return None

search_roots = [Path("/kaggle/working/repo"), Path("/kaggle/input"), Path("/kaggle/working"), Path("/content"), Path(".")]
input_script = find_train_script_in_roots(search_roots)
if input_script is None:
    raise FileNotFoundError("train_hybrid.py not found in inputs or repo.")

if (input_script.parents[2] / "backend").exists():
    SRC_ROOT = input_script.parents[2]
else:
    SRC_ROOT = input_script.parent.parent

WORK_DIR = Path("/kaggle/working/repo")
if WORK_DIR.exists():
    try:
        if not SRC_ROOT.samefile(WORK_DIR):
            shutil.rmtree(WORK_DIR)
            shutil.copytree(SRC_ROOT, WORK_DIR)
    except Exception:
        shutil.rmtree(WORK_DIR)
        shutil.copytree(SRC_ROOT, WORK_DIR)
else:
    shutil.copytree(SRC_ROOT, WORK_DIR)

candidate = WORK_DIR / "backend" / "scripts" / "train_hybrid.py"
script_path = candidate if candidate.exists() else next(WORK_DIR.rglob("train_hybrid.py"))

repo_root = WORK_DIR
script_dir = script_path.parent

# ── AUTO-FIND models/ package anywhere under /kaggle/input ──
def find_models_parent(base: Path):
    for p in base.rglob("models/__init__.py"):
        return p.parent.parent  # parent of models/ folder
    return None

models_parent = find_models_parent(Path("/kaggle/input"))
if models_parent is None:
    raise FileNotFoundError("Could not find models/__init__.py under /kaggle/input. Check your dataset is added.")

print(f"models package found at: {models_parent / 'models'}")

data_dir = Path("/kaggle/working/data").resolve()
checkpoint_dir = Path("/kaggle/working/checkpoints").resolve()
checkpoint_dir.mkdir(parents=True, exist_ok=True)

EPOCHS = 100
BATCH_SIZE = 16  # smaller batch for better gradient updates on small classes
LEARNING_RATE = 5e-5  # even lower LR
PATIENCE = 15
NUM_WORKERS = 2

env = os.environ.copy()
extra_paths = [
    str(models_parent),       # wherever models/ actually lives
    str(script_dir),
    str(repo_root),
    str(repo_root / "backend"),
]
existing = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = ":".join(extra_paths) + (":" + existing if existing else "")

cmd = [
    sys.executable, str(script_path),
    "--data-dir", str(data_dir),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--early-stopping-patience", str(PATIENCE),
    "--num-workers", str(NUM_WORKERS),
    "--checkpoint-dir", str(checkpoint_dir),
]

print("Using script:        ", script_path)
print("Using data dir:      ", data_dir)
print("Using checkpoint dir:", checkpoint_dir)
print("PYTHONPATH:          ", env["PYTHONPATH"])
print("Running:", " ".join(cmd))

subprocess.run(cmd, cwd=str(repo_root), check=True, env=env)

In [ ]:
# ...existing code...
# Zip checkpoints: prefer best_hybrid_model.pth else include all .pth
import zipfile
from pathlib import Path

ckpt_dir = Path("/kaggle/working/checkpoints") if Path("/kaggle").exists() else Path("checkpoints")
zip_path = Path("/kaggle/working/trained_model.zip") if Path("/kaggle").exists() else Path("trained_model.zip")

if not ckpt_dir.exists():
    raise FileNotFoundError(f"Checkpoint directory not found: {ckpt_dir}")

best = ckpt_dir / "best_hybrid_model.pth"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    if best.exists():
        zf.write(best, arcname=best.name)
    else:
        for p in sorted(ckpt_dir.rglob("*.pth")):
            zf.write(p, arcname=p.relative_to(ckpt_dir))

print(f"✓ Created zip: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
    print("✓ Download started (Colab).")
except Exception:
    print("ℹ️ Not in Colab. In Kaggle, download from Output -> trained_model.zip")
# ...existing code...